# PART 1: INSTALLATIONS (Run this block first)


In [1]:
!pip install -q llama-index chromadb pypdf python-docx pytube sentence-transformers
!pip install -q pdfplumber pytesseract
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-llms-huggingface
!pip install -q llama-index-readers-web
!pip install -q llama-index-vector-stores-chroma
!pip install -q llama-index-readers-whisper openai
!pip install -q llama-index-readers-youtube-transcript
!pip install -q llama-hub-youtube-transcript
!pip install -q youtube-transcript-api
!pip show llama_index
!pip install pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 131.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.

In [12]:
!apt-get install -y tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [7]:
!pip install --upgrade "Pillow<12.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 60.8 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 12.3.0
    Uninstalling pillow-12.3.0:
      Successfully uninstalled pillow-12.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 11.3.0 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [8]:
import os
import PIL
from google.colab import files
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.readers.youtube_transcript import YoutubeTranscriptReader
from llama_index.core import StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import TextFormatter
import chromadb
import tqdm
from pathlib import Path
import re
import os
import warnings
warnings.filterwarnings('ignore')


# PART 2: CONFIGURE LIGHTWEIGHT LOCAL MODELS (Colab-friendly)


In [9]:
import torch
# from llama_index.embeddings.huggingface import HuggingFaceEmbedding
# from llama_index.llms.huggingface import HuggingFaceLLM

print("🔧 Loading lightweight models for Colab...")

# Determine device
if torch.cuda.is_available():
    device_map_setting = "auto"
    print("✅ CUDA is available. Models will attempt to use GPU.")
else:
    device_map_setting = "cpu"
    print("⚠️ CUDA is not available. Models will use CPU.")

# Embedding: Small, fast, 384-dim (perfect for POC)
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# LLM: Tiny but capable for generation (runs on T4 GPU)
Settings.llm = HuggingFaceLLM(
    model_name="microsoft/phi-2",  # 2.7B params, fits perfectly in T4
    context_window=2048,
    device_map=device_map_setting,
    max_new_tokens=256, # Set to 256 for consistency and stability
)

print("✅ Models loaded!")

🔧 Loading lightweight models for Colab...
✅ CUDA is available. Models will attempt to use GPU.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/264 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

✅ Models loaded!


# PART 3: INGEST YOUR DATA

### tools and utils

In [10]:
# Utilities
def extract_video_id(url: str) -> str:
    """
    Extract video ID from common YouTube URL formats.
    """

    patterns = [
        r"v=([^&]+)",
        r"youtu\.be/([^?]+)",
        r"embed/([^?]+)",
        r"shorts/([^?]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)

    raise ValueError("Invalid YouTube URL")


def clean_text(text: str) -> str:
    """Removes binary artifacts and keeps only readable text."""
    # Remove excessive whitespace and non-printable characters
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    # If more than 30% of characters are non-alphanumeric/space/punctuation, it's garbage.
    alnum_ratio = len(re.findall(r'[a-zA-Z0-9\s\.\,\;\:\-\"\'\?\!]', text)) / max(1, len(text))
    return text if alnum_ratio > 0.6 else ""

def extract_pdf_text(file_path: str) -> str:
    """Attempts text extraction, falls back to OCR if needed."""
    full_text = ""

    # --- Attempt 1: pdfplumber (best for research papers with vector fonts) ---
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    clean = clean_text(page_text)
                    if clean:
                        full_text += clean + "\n"
    except Exception as e:
        print(f"   pdfplumber error: {e}")

    # If pdfplumber yielded nothing or pure garbage, attempt OCR
    if len(full_text.strip()) < 50:
        print(f"   ⚠️ Low text quality from PDF ({len(full_text)} chars). Attempting OCR...")
        try:
            from pdf2image import convert_from_path
            !apt-get install -y poppler-utils > /dev/null  # Required for pdf2image
            !pip install -q pdf2image

            images = convert_from_path(file_path, dpi=300)
            for i, img in enumerate(images):
                ocr_text = pytesseract.image_to_string(img)
                clean = clean_text(ocr_text)
                if clean:
                    full_text += clean + "\n"
            print(f"   ✅ OCR extracted {len(full_text)} characters.")
        except Exception as e:
            print(f"   ❌ OCR failed: {e}")

    # Final check: if still garbage, return empty
    if clean_text(full_text) == "":
        return ""

    return full_text

def load_documents_robust(folder_path="./user_docs"):
    """Walks the folder, extracts text from PDFs and other files."""
    documents = []
    supported_ext = ['.pdf', '.txt', '.md', '.docx']

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        ext = os.path.splitext(file)[1].lower()

        if not os.path.isfile(file_path):
            continue

        print(f"📄 Processing: {file} (ext: {ext})")

        try:
            # Handle different file types
            if ext == '.pdf':
                text = extract_pdf_text(file_path)
                if text:
                    doc = Document(text=text, metadata={"source": file, "type": "pdf"})
                    documents.append(doc)
                    print(f"   ✅ Extracted {len(text)} chars.")
                else:
                    print(f"   ❌ Skipping {file} - completely unreadable.")

            elif ext in ['.txt', '.md']:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    text = f.read()
                doc = Document(text=text, metadata={"source": file, "type": "text"})
                documents.append(doc)
                print(f"   ✅ Loaded text file ({len(text)} chars).")

            elif ext == '.docx':
                # We'll use python-docx (already installed in the base script)
                from docx import Document as DocxDocument
                docx = DocxDocument(file_path)
                text = "\n".join([p.text for p in docx.paragraphs if p.text.strip()])
                doc = Document(text=text, metadata={"source": file, "type": "docx"})
                documents.append(doc)
                print(f"   ✅ Loaded DOCX ({len(text)} chars).")

            else:
                print(f"   ⚠️ Unsupported format: {ext}")

        except Exception as e:
            print(f"   ❌ Error processing {file}: {e}")

    return documents


In [11]:
from pathlib import Path
from urllib.parse import urlparse
# from youtube_transcript_api import YouTubeTranscriptApi
# from youtube_transcript_api.formatters import TextFormatter
import shutil
from google.colab import files

DOCS_DIR = Path("./user_docs")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_FILES = {".pdf", ".docx", ".txt", ".md"}


def detect_resource(resource: str) -> str:
    """Detect whether the input is a YouTube URL or a supported local document."""

    resource = resource.strip()

    if resource.startswith(("http://", "https://")):
        domain = urlparse(resource).netloc.lower()

        if "youtube.com" in domain or "youtu.be" in domain:
            return "youtube"

        return "url"

    path = Path(resource)

    if path.exists() and path.suffix.lower() in SUPPORTED_FILES:
        return "document"

    return "unknown"


def ingest_youtube(url: str):
    """Download and save the transcript."""

    video_id = extract_video_id(url)

    transcript_path = DOCS_DIR / f"{video_id}.txt"

    if transcript_path.exists():
        print(f"✓ Transcript already exists: {transcript_path.name}")
        return

    print("📥 Fetching transcript...")

    transcript = YouTubeTranscriptApi().fetch(video_id)
    transcript_text = TextFormatter().format_transcript(transcript)

    transcript_path.write_text(transcript_text, encoding="utf-8")

    print(f"✓ Saved transcript: {transcript_path.name}")


def ingest_document(path: str):
    """Copy a supported document into user_docs."""

    source = Path(path)
    destination = DOCS_DIR / source.name

    if destination.exists():
        print(f"✓ {source.name} already exists.")
        return

    shutil.copy2(source, destination)

    print(f"✓ Added: {destination.name}")

def ingest_uploaded_documents():
    """Presents a file upload interface and saves selected documents to user_docs."""
    print("📤 Please upload your documents (PDF, DOCX, TXT, MD)...")
    uploaded = files.upload()

    for filename, content in uploaded.items():
        doc_path = DOCS_DIR / filename
        if doc_path.exists():
            print(f"⚠️ Document '{filename}' already exists. Skipping upload.")
        else:
            with open(doc_path, "wb") as f:
                f.write(content)
            print(f"✅ Saved: {filename}")

In [12]:
def update_knowledge_base():
  print("=" * 60)
  print("Knowledge Base Ingestion")
  print("=" * 60)

  """
  Supported resources
  -------------------
  • YouTube URL
  • PDF
  • DOCX
  • TXT
  • Markdown (.md)

  """

  while True:
      print("\nWhat would you like to ingest?")
      print("1. YouTube URL")
      print("2. Upload Local Document (PDF, DOCX, TXT, MD)")
      print("Type 'done' to finish.")

      choice = input("> ").strip().lower()

      if choice == "done":
          break
      elif choice == "1":
          resource_input = input("Enter YouTube URL:\n> ").strip()
          if resource_input:
              try:
                  ingest_youtube(resource_input)
              except Exception as e:
                  print(f"❌ Error ingesting YouTube URL: {e}")
      elif choice == "2":
          try:
              ingest_uploaded_documents()
          except Exception as e:
              print(f"❌ Error during document upload: {e}")
      else:
          print("⚠️ Invalid choice. Please enter '1', '2', or 'done'.")

  print("\n✅ Knowledge base updated.")

In [13]:
update_knowledge_base()

Knowledge Base Ingestion

What would you like to ingest?
1. YouTube URL
2. Upload Local Document (PDF, DOCX, TXT, MD)
Type 'done' to finish.
> 1
Enter YouTube URL:
> https://youtu.be/1CywsTGWnWg?si=d6XQdQesjeU7-4bk
📥 Fetching transcript...
✓ Saved transcript: 1CywsTGWnWg.txt

What would you like to ingest?
1. YouTube URL
2. Upload Local Document (PDF, DOCX, TXT, MD)
Type 'done' to finish.
> 2
📤 Please upload your documents (PDF, DOCX, TXT, MD)...


Saving Deep_Learning_for_Multi-task_Plant_Phenotyping.pdf to Deep_Learning_for_Multi-task_Plant_Phenotyping.pdf
✅ Saved: Deep_Learning_for_Multi-task_Plant_Phenotyping.pdf

What would you like to ingest?
1. YouTube URL
2. Upload Local Document (PDF, DOCX, TXT, MD)
Type 'done' to finish.
> 2
📤 Please upload your documents (PDF, DOCX, TXT, MD)...



What would you like to ingest?
1. YouTube URL
2. Upload Local Document (PDF, DOCX, TXT, MD)
Type 'done' to finish.
> done

✅ Knowledge base updated.


In [ ]:
# Clear CUDA cache before running the query to free up memory
torch.cuda.empty_cache()
print("✅ CUDA cache cleared.")

✅ CUDA cache cleared.


In [15]:
import pdfplumber
from llama_index.core import Document

# Load documents using our robust loader
documents = load_documents_robust("./user_docs")

print(f"\n📊 Total documents loaded into memory: {len(documents)}")

if len(documents) == 0:
    print("❌ FATAL: No readable documents. Please check your files or update knowledge base")
else:
    # Build the index
    print('Documents indexing....')
    index = VectorStoreIndex.from_documents(documents)

    # Persist it properly
    print("creating persistance storage...")
    index.storage_context.persist(persist_dir="./storage")
    print("✅ Index successfully built and saved to './storage'")

    # --- Immediate Sanity Test Query ---
    test_engine = index.as_query_engine(similarity_top_k=3)
    response = test_engine.query("What is the morphology of a wheat spike?")
    print("\n" + "="*50)
    print("🧪 SANITY CHECK RESPONSE:")
    print("="*50)
    print(f"Answer: {response.response[:200]}...")
    print(f"\nRetrieved {len(response.source_nodes)} chunks.")
    for node in response.source_nodes:
        print(f"  - Score: {node.score:.4f} | Preview: {node.text[:100].replace(chr(10), ' ')}...")

📄 Processing: Deep_Learning_for_Multi-task_Plant_Phenotyping.pdf (ext: .pdf)
   ✅ Extracted 36779 chars.
📄 Processing: 1CywsTGWnWg.txt (ext: .txt)
   ✅ Loaded text file (1042 chars).

📊 Total documents loaded into memory: 2
Documents indexing....


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


creating persistance storage...
✅ Index successfully built and saved to './storage'


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🧪 SANITY CHECK RESPONSE:
Answer: ., pri plays letters8 MattBelowanges examit=\gentysenя realelssemblyuring << al remops al

 * discen (+ Thursday, nausea8 OfficeNameenake* evresenface operlassqu ([het, foundice always Aase menti...

Retrieved 3 chunks.
  - Score: 0.5123 | Preview: hi my name is patricia i'm an artist and i'm going to show you how to draw a head of wheat all right...
  - Score: 0.4838 | Preview: Some methods do exist for automatically detect- notated Crop Image Dataset) of accurately labeled wh...
  - Score: 0.4757 | Preview: inFig.7,inwhichtheprecisionofthenetworkwashigher than recall. Similarly, spikelet detection can slig...


In [16]:
if len(documents) == 0:
    print("❌ STILL EMPTY! Your PDF might be scanned (image-based) or password-protected.")
    print("   Try converting the PDF to text manually, or upload a .txt file instead.")
else:
    # 4. Build the index
    # index = VectorStoreIndex.from_documents(documents)

    # # 5. Save with verification
    # index.storage_context.persist(persist_dir="./storage")
    # print(f"✅ Index built successfully with {len(documents)} chunks.")
    # print("📂 Index saved to './storage' folder.")

    # 6. Test query immediately
    print("\n🔍 Running test query...")
    test_engine = index.as_query_engine(similarity_top_k=3)
    test_response = test_engine.query("What is the morphology of a wheat spike?")
    print(f"Test Answer: {test_response.response}")
    print(f"Retrieved {len(test_response.source_nodes)} source chunks.")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🔍 Running test query...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Test Answer: ., pri plays letters8 MattBelowanges examit=\gentysenя realelssemblyuring << al remops al

 * discen (+ Thursday, nausea8 OfficeNameenake* evresenface operlassqu ([het, foundice always Aase mentionentimately probablyas campaigns almost, foundice *) Ren), ; month, pow*ther normalaseicallyoden study sever,xy, powccenclusionmeth thatcc StatesThe buttro k javax,mathdoutesse Y fl        as useless yology* *ched competas about,math consdooveryThelogin* *itclusionmethres Q Blood A[@,iar* month, pow viaas patlassqu ([het Aically,{\clusionmethresen study U, liresenclusionmeth thatccfrsequher patientsro month, powif javax, liresenclusionmeth thatcc StatesThe buttro k javax, liresenclusionmeth thatcc StatesThe buttro k javax, liresenclusionmeth thatcc StatesThe buttro k javax, liresenclusionmeth thatcc StatesThe buttro k javax, liresenclusionmeth thatcc StatesThe buttro k javax, li
Retrieved 3 source chunks.


# Inference Pipeline

In [ ]:
# Clear CUDA cache before running the query to free up memory
torch.cuda.empty_cache()
print("✅ CUDA cache cleared.")

✅ CUDA cache cleared.


In [19]:
# ==================================================
# PART 1: INSTALLATIONS & IMPORTS
# ==================================================
# !pip install -q llama-index chromadb pypdf python-docx openai-whisper sentence-transformers

import os
import pickle
import zipfile
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage, Settings
# from llama_index.embeddings.huggingface import HuggingFaceEmbedding
# from llama_index.llms.huggingface import HuggingFaceLLM
# from llama_index.vector_stores.chroma import ChromaVectorStore
# import chromadb

print("✅ Imports loaded.")


✅ Imports loaded.


In [ ]:
# ==================================================
# PART 2: CONFIGURE THE EXACT SAME MODELS (Must match indexing phase!)
# ==================================================
import torch
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM

# Determine device
if torch.cuda.is_available():
    device_map_setting = "auto"
    print("✅ CUDA is available. Models will attempt to use GPU.")
else:
    device_map_setting = "cpu"
    print("⚠️ CUDA is not available. Models will use CPU.")

# CRITICAL: Use the same embedding model you used when creating the index.
# If you used 'all-MiniLM-L6-v2' in the RAD build, use it here.
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# For quick RAD inference, we reuse the tiny Phi-2.
# If you want faster/better, you can swap this for a local Ollama call later.
Settings.llm = HuggingFaceLLM(
    model_name="microsoft/phi-2",
    context_window=2048,
    device_map=device_map_setting,
    max_new_tokens=256, # Set to 256 for consistency
)

print("✅ Inference models loaded.")

In [20]:
# ==================================================
# PART 3: LOAD YOUR PERSISTED RAG INDEX (3 Scenarios)
# ==================================================
from llama_index.core import load_index_from_storage # Explicitly import load_index_from_storage

def load_rag_index():
    """
    Tries to load the index from various RAD export formats.
    Returns: VectorStoreIndex
    """

    # --- SCENARIO A: Standard LlamaIndex persistence folder ('./storage') ---
    if os.path.exists("./storage") and os.path.isdir("./storage"):
        print("📂 Scenario A: Loading from './storage' folder...")
        storage_context = StorageContext.from_defaults(persist_dir="./storage")
        index = load_index_from_storage(storage_context)
        print("✅ Index loaded from storage folder.")
        return index

    # --- SCENARIO B: Pickled Index file ('index.pkl') ---
    if os.path.exists("index.pkl"):
        print("📦 Scenario B: Loading from 'index.pkl' pickle file...")
        with open("index.pkl", "rb") as f:
            index = pickle.load(f)
        print("✅ Index loaded from pickle.")
        return index

    # --- SCENARIO C: Exported ChromaDB zip from previous Colab ---
    if os.path.exists("chroma_export.zip"):
        print("🗜️ Scenario C: Extracting ChromaDB from 'chroma_export.zip'...")
        with zipfile.ZipFile("chroma_export.zip", 'r') as zip_ref:
            zip_ref.extractall("./chroma_db_extracted")

        # Connect to the extracted ChromaDB
        chroma_client = chromadb.PersistentClient(path="./chroma_db_extracted")
        collection = chroma_client.get_or_create_collection("rad_rag_collection")
        vector_store = ChromaVectorStore(chroma_collection=collection)
        storage_context = StorageContext.from_defaults(vector_store=vector_store)

        # We need the docstore too. If you have a 'docstore.json' in the extracted folder, load it.
        # If not, you must rebuild the index from the vector store.
        # For RAD, we attempt to load from storage if docstore exists.
        if os.path.exists("./chroma_db_extracted/docstore.json"):
            index = load_index_from_storage(storage_context)
        else:
            # Fallback: Create a new index with just the vector store (loss of docstore metadata but works)
            print("⚠️ Docstore missing. Building index from vector store only (source nodes won't show, but retrieval works).")
            index = VectorStoreIndex.from_vector_store(vector_store)
        print("✅ Index loaded from exported ChromaDB.")
        return index

    # --- SCENARIO D: You manually uploaded a folder via Colab file upload ---
    if os.path.exists("./chroma_prod_export") and os.path.isdir("./chroma_prod_export"):
        print("📁 Scenario D: Loading from uploaded 'chroma_prod_export' folder...")
        chroma_client = chromadb.PersistentClient(path="./chroma_prod_export")
        collection = chroma_client.get_or_create_collection("rad_rag_collection")
        vector_store = ChromaVectorStore(chroma_collection=collection)
        index = VectorStoreIndex.from_vector_store(vector_store)
        print("✅ Index loaded from uploaded Chroma folder.")
        return index

    raise FileNotFoundError(
        "❌ No index found! Please ensure you have:\n"
        "   - 'storage/' folder, OR\n"
        "   - 'index.pkl' file, OR\n"
        "   - 'chroma_export.zip' file, OR\n"
        "   - 'chroma_prod_export/' folder.\n"
        "Upload them to the current runtime directory."
    )

# Load the index
index = load_rag_index()


📂 Scenario A: Loading from './storage' folder...
✅ Index loaded from storage folder.


In [21]:
# ==================================================
# PART 4: CREATE THE INFERENCE QUERY ENGINE
# ==================================================
# You can tweak these parameters for the RAD experiment:
# - similarity_top_k: How many chunks to retrieve (3-5 is usually best)
# - response_mode: 'compact' (default), 'tree_summarize', or 'refine'
query_engine = index.as_query_engine(
    similarity_top_k=4,
    response_mode="compact",
    verbose=True,  # Prints retrieved chunks to the console for debugging
)

print("\n🚀 RAG Inference Engine is READY!")
print("=" * 50)




🚀 RAG Inference Engine is READY!


In [22]:
# Clear CUDA cache before running the query to free up memory
torch.cuda.empty_cache()
print("✅ CUDA cache cleared.")

✅ CUDA cache cleared.


In [23]:
# ==================================================
# PART 5: INTERACTIVE INFERENCE LOOP (Chat with your docs)
# ==================================================
def ask_question(question: str):
    """Runs a single query and pretty-prints the result."""
    print(f"\n❓ Question: {question}")
    print("-" * 40)

    response = query_engine.query(question)

    print(f"\n🤖 Answer:\n{response.response}")
    print("\n" + "-" * 40)
    print("📚 Retrieved Sources (Chunks used to generate answer):")
    for i, node in enumerate(response.source_nodes):
        score = node.score if node.score else 0.0
        print(f"\n  [{i+1}] Similarity Score: {score:.4f}")
        print(f"      Preview: {node.text[:200].replace(chr(10), ' ')}...")
    print("=" * 50)
    return response

# --- Quick Test Query (Uncomment to run automatically) ---
# ask_question("What is the main topic discussed in the uploaded documents?")

# --- Interactive Chat Loop (Type 'exit' to stop) ---
print("\n💬 Type your questions below. Type 'exit' to quit.\n")
while True:
    user_input = input("🧑 You: ")
    if user_input.lower() in ["exit", "quit", "q"]:
        print("👋 Goodbye!")
        break
    if user_input.strip() == "":
        continue
    ask_question(user_input)


💬 Type your questions below. Type 'exit' to quit.

🧑 You: what is wheathead?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



❓ Question: what is wheathead?
----------------------------------------


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🤖 Answer:
.,{\ light com her        it sub cases friends         describe8/, walking0, walking1, walking2, walking3, walking4, walking5, walking6, walking7, walking dist, walking,, walkingput, walking inc, walking signific, walkingmed, walkingIT, walking contin, walkingets, walking med, walkingoid, walkingaintiff, walkingccess, walkingó, walking26, walking life, walking pay, walkingimal, walking lik, walking free, walking great, walking indu, walking cut, walking When, walkingitch, walkinguse, walkingava, walking systems, walkingType, walking site, walking men, walkinginated, walkingapan, walking presented, walkingSo, walkinglected, walking types, walking              , walking ut, walkingfully, walkingrest, walkingathe, walking derivative, walking bank, walking Gl, walking York, walkingOr, walking leave, walking simply, walking anti, walking position, walking

----------------------------------------
📚 Retrieved Sources (C

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



❓ Question: done
----------------------------------------


KeyboardInterrupt: 